# MERLIN: second-order routing for multi-state linear attention
Companion notebook for the 2026-10-30 post. Paper: Hwang et al., "Multi-State Linear Attention with Generalized Second-Order Routing", arXiv:2610.37645 (Oct 27, 2026).
We reproduce the core idea in miniature: two segments with identical mean alignment but different variance, and show mean-only routing splits 50/50 while Merlin picks the needle.

In [ ]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)

# Toy segment alignments with a query: segment A has one sharp match (0.9) + weak rest;
# segment B is uniformly mediocre. Both have mean exactly 0.20.
A = np.array([0.9] + [0.1]*7)
B = np.array([0.24, 0.16]*4)
print("mean A:", A.mean(), " mean B:", B.mean())
print("var  A:", A.var(), " var  B:", B.var())
assert abs(A.mean() - B.mean()) < 1e-12
assert A.var() / B.var() > 40  # 43.75x spread

def softmax(x, tau=1.0):
    e = np.exp(tau*(x - x.max())); return e/e.sum()

mean_only = softmax(np.array([A.mean(), B.mean()]), tau=5.0)
# Merlin: gated mixture of mean-softmax and variance-softmax (gate 0.5, temps 5 and 30)
merlin = 0.5*softmax(np.array([A.mean(), B.mean()]), tau=5.0) \
       + 0.5*softmax(np.array([A.var(), B.var()]), tau=30.0)
print("mean-only weights:", mean_only)
print("Merlin weights:   ", merlin)
assert abs(mean_only[0] - 0.5) < 1e-9, "mean-only cannot distinguish the segments"
assert merlin[0] > 0.65, "Merlin must favor the sharp segment"
print("OK: mean-only splits 50/50; Merlin gives the needle segment %.1f%%" % (100*merlin[0]))


## 2. Why variance appears: the grouped-softmax expansion
Group full softmax attention mass by segment. The log-evidence for segment i expands in cumulants:
log Z = delta * mean + delta^2/2 * variance + O(delta^3). Mean-only routing keeps the first term; Merlin keeps both. Here we check that the second-order approximation tracks true grouped softmax mass better than the first-order one on a random segment.

In [ ]:
rng = np.random.default_rng(7)
# one segment of 64 token-level scores: a few sharp matches in noise
scores = np.concatenate([rng.normal(0.1, 0.05, 56), [0.9, 0.85, 0.8, 0.75, 0.7, 0.65, 0.6, 0.55]])
delta = 2.0
true_logZ = np.log(np.mean(np.exp(delta*scores)))          # exact grouped log-evidence
first  = delta*scores.mean()                                # first-order only
second = delta*scores.mean() + (delta**2/2)*scores.var()    # Merlin's two terms
print("exact log-evidence: %.4f" % true_logZ)
print("first-order approx: %.4f  (err %.4f)" % (first, abs(first-true_logZ)))
print("second-order approx: %.4f  (err %.4f)" % (second, abs(second-true_logZ)))
assert abs(second-true_logZ) < abs(first-true_logZ)
print("OK: second-order approximation is closer to true grouped softmax mass")


## 3. Paper headline numbers, recomputed
RULER @16K (avg) and LongBench (avg) from Tables 2-3 of arXiv:2610.37645. Relative gains are vs Meanpool, the strongest baseline.

In [ ]:
ruler = {"GDN": (62.7, 57.9), "Mamba-2": (76.2, 71.2)}
longbench = {"GDN": (21.9, 19.3), "Mamba-2": (20.6, 18.4)}
for name, (m, b) in ruler.items():
    print(f"RULER {name}: Merlin {m} vs Meanpool {b} -> +{100*(m/b-1):.1f}%")
for name, (m, b) in longbench.items():
    print(f"LongBench {name}: Merlin {m} vs Meanpool {b} -> +{100*(m/b-1):.1f}%")
assert abs(100*(62.7/57.9-1) - 8.3) < 0.1
assert abs(100*(76.2/71.2-1) - 7.0) < 0.1
assert abs(100*(21.9/19.3-1) - 13.5) < 0.1
assert abs(100*(20.6/18.4-1) - 12.0) < 0.1
# Needle task spotlight: S-NIAH-3
print("S-NIAH-3 GDN: 21.2 -> 47.0 (2.22x); Mamba-2: 66.2 -> 96.0")
assert abs(47.0/21.2 - 2.22) < 0.01
print("OK: all headline gains recompute")


## 4. Memory scaling: why multi-state exists
Uniform 256-token segments: a 16K context becomes 64 segments. Full attention's KV cache grows with every token; a single linear state is flat but lossy; multi-state grows one fixed-size state per segment.

In [ ]:
for ctx in [4096, 16384, 65536, 262144]:
    n_seg = ctx // 256
    print(f"context {ctx:>7,} tokens -> {n_seg:>4} segments (256 tokens each)")
assert 16384 // 256 == 64
print("OK: 16K context = 64 segments, as in the paper's setup")
